In [1]:
import duckdb
con = duckdb.connect("olist.duckdb")

In [2]:
con.execute("""
SELECT COUNT(*) AS rows_after_bad_join,
       ROUND(SUM(oi.price), 2) AS inflated_revenue
FROM orders o
JOIN order_items oi USING (order_id)
JOIN order_payments p USING (order_id)
""").df()

,rows_after_bad_join,inflated_revenue
0,117601,14209115.34


In [3]:
con.execute("""
CREATE OR REPLACE TABLE order_level AS
WITH items AS (
  SELECT order_id,
         SUM(price) AS item_revenue,
         SUM(freight_value) AS freight,
         COUNT(*) AS n_items
  FROM order_items
  GROUP BY order_id
),
pay AS (
  SELECT order_id, SUM(payment_value) AS paid
  FROM order_payments
  GROUP BY order_id
),
rev AS (
  SELECT order_id, AVG(review_score) AS review_score
  FROM order_reviews
  GROUP BY order_id
)
SELECT o.*,
       c.customer_unique_id,
       c.customer_state,
       i.item_revenue,
       i.freight,
       i.n_items,
       p.paid,
       r.review_score
FROM orders o
LEFT JOIN customers c USING (customer_id)
LEFT JOIN items i USING (order_id)
LEFT JOIN pay p USING (order_id)
LEFT JOIN rev r USING (order_id)
""")
print("order_level created")

order_level created


In [4]:
con.execute("""
CREATE OR REPLACE TABLE item_level AS
SELECT oi.*,
       p.product_category_name,
       t.product_category_name_english AS category,
       o.order_status,
       o.order_purchase_timestamp
FROM order_items oi
LEFT JOIN products p USING (product_id)
LEFT JOIN category_translation t USING (product_category_name)
LEFT JOIN orders o USING (order_id)
""")
print("item_level created")

item_level created


In [5]:
con.execute("""
SELECT
  (SELECT COUNT(*) FROM orders)      AS orders_rows,
  (SELECT COUNT(*) FROM order_level) AS order_level_rows,
  (SELECT COUNT(*) FROM order_items) AS items_rows,
  (SELECT COUNT(*) FROM item_level)  AS item_level_rows
""").df()

,orders_rows,order_level_rows,items_rows,item_level_rows
0,99441,99441,112650,112650


In [6]:
con.execute("""
SELECT
  (SELECT ROUND(SUM(price), 2)        FROM order_items) AS source_revenue,
  (SELECT ROUND(SUM(item_revenue), 2) FROM order_level) AS order_level_revenue,
  (SELECT ROUND(SUM(price), 2)        FROM item_level)  AS item_level_revenue
""").df()

,source_revenue,order_level_revenue,item_level_revenue
0,13591643.7,13591643.7,13591643.7


In [7]:
con.execute("""
SELECT
  (SELECT ROUND(SUM(freight_value), 2) FROM order_items) AS source_freight,
  (SELECT ROUND(SUM(freight), 2)       FROM order_level) AS order_level_freight
""").df()

,source_freight,order_level_freight
0,2251909.54,2251909.54


In [8]:
con.execute("""
SELECT COUNT(*) AS rows,
       COUNT(DISTINCT order_id) AS distinct_orders
FROM order_level
""").df()

,rows,distinct_orders
0,99441,99441


In [9]:
con.execute("SELECT * FROM order_level LIMIT 5").df()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_state,item_revenue,freight,n_items,paid,review_score
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,SP,29.99,8.72,1,38.71,4.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,BA,118.70,22.76,1,141.46,4.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,GO,159.90,19.22,1,179.12,5.0
3,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,SP,19.90,8.72,1,28.62,5.0
4,a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,delivered,2017-07-09 21:57:05,2017-07-09 22:10:13,2017-07-11 14:58:04,2017-07-26 10:57:55,2017-08-01,80bb27c7c16e8f973207a5086ab329e2,PR,147.90,27.36,1,175.26,4.0


In [10]:
con.execute("""
SELECT
  COUNT(*)                             AS orders,
  COUNT(item_revenue)                  AS with_items,
  COUNT(review_score)                  AS with_review,
  COUNT(order_delivered_customer_date) AS with_delivery_date
FROM order_level
""").df()

,orders,with_items,with_review,with_delivery_date
0,99441,98666,98673,96476
